# N0: Error Analysis - Model Prediction Errors

Analyze where the best model makes mistakes on validation set:
1. Load validation set
2. Extract TF-IDF & PhoBERT embeddings
3. Load best model from model/machine_learned
4. Test on validation set
5. Extract and analyze incorrect predictions

## 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import warnings
import torch
import gc
import joblib
import os
from tqdm import tqdm
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score, confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModel

# Setup device for PyTorch (PhoBERT extraction)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("✅ All imports successful")
print(f"   PyTorch Device: {device}")
print(f"   Mode: Error Analysis on Validation Set")

## 2. Load Validation Set

In [ ]:
val_path = '../../data/splited/val_set.csv'

df_val = pd.read_csv(val_path)
y_val = df_val['label'].values

print(f"✅ Validation set loaded: {df_val.shape}")
print(f"   Labels: {(y_val==0).sum()} fake / {(y_val==1).sum()} real")
print(f"\n   Columns: {df_val.columns.tolist()}")

## 3. Generate TF-IDF Embeddings

In [ ]:
texts_val_strict = df_val['text_strict'].fillna('').tolist()

# Use same TF-IDF parameters as training
tfidf = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=2, sublinear_tf=True)
X_val_tfidf_full = tfidf.fit_transform(texts_val_strict)

svd = TruncatedSVD(n_components=900, random_state=42)
X_val_tfidf = svd.fit_transform(X_val_tfidf_full)

print(f"✅ TF-IDF embeddings: Val {X_val_tfidf.shape}")

## 4. Extract PhoBERT Embeddings

In [ ]:
texts_val_loose = df_val['text_loose'].fillna('').tolist()

def extract_phobert_embeddings(texts, batch_size=16):
    """Extract PhoBERT embeddings from text"""
    tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')
    model = AutoModel.from_pretrained('vinai/phobert-base-v2').to(device).eval()
    embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="PhoBERT", leave=False):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=256)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs, output_hidden_states=True)
            cls_tokens = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.extend(cls_tokens)
            del outputs, inputs
            torch.cuda.empty_cache()
    
    model.to('cpu')
    del model, tokenizer
    gc.collect()
    return np.array(embeddings)

X_val_phobert = extract_phobert_embeddings(texts_val_loose, batch_size=16)

print(f"✅ PhoBERT embeddings: Val {X_val_phobert.shape}")

## 5. Combine and Scale Embeddings

In [ ]:
# Combine embeddings (TF-IDF + PhoBERT)
X_val_combined = np.hstack([X_val_tfidf, X_val_phobert])

print(f"✅ Combined embeddings: {X_val_combined.shape[1]} dimensions")
print(f"   TF-IDF: 900 + PhoBERT: 768 = Total: {900 + 768}")

## 6. Load Best Model & Scaler

In [ ]:
print("\n" + "="*80)
print("LOADING BEST MODEL")
print("="*80)

# Find model files in model/machine_learned
model_dir = os.path.abspath('../../model/machine_learned')
print(f"\n📂 Model directory: {model_dir}")

# List files in directory
if os.path.exists(model_dir):
    files = os.listdir(model_dir)
    print(f"\n📁 Files found:")
    for f in sorted(files):
        print(f"   - {f}")
else:
    print(f"❌ Directory not found: {model_dir}")
    raise FileNotFoundError(f"Model directory not found: {model_dir}")

# Load the most recent model and scaler
model_files = [f for f in files if f.endswith('.pkl') and not f.startswith('scaler')]
if not model_files:
    raise FileNotFoundError("No model files found in machine_learned directory")

# Get the most recent model file (alphabetically, assuming YYYYMMDD_HHMMSS format)
latest_model_file = sorted(model_files)[-1]
model_path = os.path.join(model_dir, latest_model_file)

# Find corresponding scaler
timestamp = latest_model_file.split('_')[-1].replace('.pkl', '')
scaler_file = f'scaler_{timestamp}.pkl'
scaler_path = os.path.join(model_dir, scaler_file)

print(f"\n🔄 Loading model: {latest_model_file}")
best_model = joblib.load(model_path)
print(f"✅ Model loaded successfully")

print(f"\n🔄 Loading scaler: {scaler_file}")
scaler = joblib.load(scaler_path)
print(f"✅ Scaler loaded successfully")

## 7. Scale and Predict on Validation Set

In [ ]:
print("\n" + "="*80)
print("MAKING PREDICTIONS ON VALIDATION SET")
print("="*80)

# Scale validation data
X_val_combined_scaled = scaler.transform(X_val_combined)
print(f"\n✅ Data scaled: {X_val_combined_scaled.shape}")

# Make predictions
y_val_pred = best_model.predict(X_val_combined_scaled)
y_val_proba = best_model.predict_proba(X_val_combined_scaled)[:, 1]

print(f"✅ Predictions made for {len(y_val_pred)} samples")
print(f"\n📊 Prediction Distribution:")
print(f"   Fake (0): {(y_val_pred==0).sum()}")
print(f"   Real (1): {(y_val_pred==1).sum()}")

## 8. Calculate Metrics

In [ ]:
print("\n" + "="*80)
print("OVERALL PERFORMANCE METRICS")
print("="*80)

f1 = f1_score(y_val, y_val_pred, average='weighted')
auc = roc_auc_score(y_val, y_val_proba)
acc = accuracy_score(y_val, y_val_pred)
prec = precision_score(y_val, y_val_pred, average='weighted')
rec = recall_score(y_val, y_val_pred, average='weighted')

print(f"\n✅ Performance Metrics:")
print(f"   F1 Score:    {f1:.4f}")
print(f"   AUC-ROC:     {auc:.4f}")
print(f"   Accuracy:    {acc:.4f}")
print(f"   Precision:   {prec:.4f}")
print(f"   Recall:      {rec:.4f}")

# Confusion Matrix
cm = confusion_matrix(y_val, y_val_pred)
print(f"\n📊 Confusion Matrix:")
print(f"   True Negatives (TN):  {cm[0,0]}  |  False Positives (FP): {cm[0,1]}")
print(f"   False Negatives (FN): {cm[1,0]}  |  True Positives (TP):  {cm[1,1]}")

# Classification Report
print(f"\n📋 Classification Report:")
print(classification_report(y_val, y_val_pred, target_names=['Fake (0)', 'Real (1)']))

## 9. Identify Incorrect Predictions

In [ ]:
print("\n" + "="*80)
print("ERROR ANALYSIS: INCORRECT PREDICTIONS")
print("="*80)

# Find incorrect predictions
incorrect_mask = y_val != y_val_pred
incorrect_indices = np.where(incorrect_mask)[0]

print(f"\n📊 Error Statistics:")
print(f"   Total samples: {len(y_val)}")
print(f"   Correct predictions: {(~incorrect_mask).sum()}")
print(f"   Incorrect predictions: {incorrect_mask.sum()}")
print(f"   Error rate: {incorrect_mask.sum() / len(y_val) * 100:.2f}%")

# Analyze error types
false_positives_mask = (y_val == 0) & (y_val_pred == 1)  # Real labeled as Fake
false_negatives_mask = (y_val == 1) & (y_val_pred == 0)  # Fake labeled as Real

print(f"\n❌ Error Types:")
print(f"   False Positives (Real → Fake): {false_positives_mask.sum()}")
print(f"   False Negatives (Fake → Real): {false_negatives_mask.sum()}")

## 10. Analyze False Positives (Real labeled as Fake)

In [ ]:
print("\n" + "="*80)
print("FALSE POSITIVES: Real News Predicted as Fake")
print("="*80)

fp_indices = np.where(false_positives_mask)[0]
print(f"\n📊 Total False Positives: {len(fp_indices)}")

if len(fp_indices) > 0:
    # Sample up to 10 false positives
    sample_size = min(10, len(fp_indices))
    sampled_fp_indices = np.random.choice(fp_indices, size=sample_size, replace=False)
    
    print(f"\n📝 Sample False Positives ({sample_size} examples):")
    print("="*80)
    
    for idx, i in enumerate(sampled_fp_indices, 1):
        print(f"\n▶ Example {idx}:")
        print(f"   ID: {df_val.iloc[i]['id']}")
        print(f"   Actual Label: Real (1)")
        print(f"   Predicted: Fake (0)")
        print(f"   Confidence: {1 - y_val_proba[i]:.4f}")
        print(f"   Text: {df_val.iloc[i]['text_strict'][:200]}...")
        
else:
    print(f"\n✅ No False Positives! All real news correctly classified.")

## 11. Analyze False Negatives (Fake labeled as Real)

In [ ]:
print("\n" + "="*80)
print("FALSE NEGATIVES: Fake News Predicted as Real")
print("="*80)

fn_indices = np.where(false_negatives_mask)[0]
print(f"\n📊 Total False Negatives: {len(fn_indices)}")

if len(fn_indices) > 0:
    # Sample up to 10 false negatives
    sample_size = min(10, len(fn_indices))
    sampled_fn_indices = np.random.choice(fn_indices, size=sample_size, replace=False)
    
    print(f"\n📝 Sample False Negatives ({sample_size} examples):")
    print("="*80)
    
    for idx, i in enumerate(sampled_fn_indices, 1):
        print(f"\n▶ Example {idx}:")
        print(f"   ID: {df_val.iloc[i]['id']}")
        print(f"   Actual Label: Fake (0)")
        print(f"   Predicted: Real (1)")
        print(f"   Confidence: {y_val_proba[i]:.4f}")
        print(f"   Text: {df_val.iloc[i]['text_strict'][:200]}...")
        
else:
    print(f"\n✅ No False Negatives! All fake news correctly classified.")

## 12. Summary of Error Patterns

In [ ]:
print("\n" + "="*80)
print("ERROR ANALYSIS SUMMARY")
print("="*80)

print(f"\n📊 Overall Performance:")
print(f"   ✅ Correct: {(~incorrect_mask).sum()} / {len(y_val)} ({(~incorrect_mask).sum() / len(y_val) * 100:.2f}%)")
print(f"   ❌ Incorrect: {incorrect_mask.sum()} / {len(y_val)} ({incorrect_mask.sum() / len(y_val) * 100:.2f}%)")

print(f"\n📈 Error Breakdown:")
print(f"   False Positives (Real → Fake): {false_positives_mask.sum()} ({false_positives_mask.sum() / len(y_val) * 100:.2f}%)")
print(f"   False Negatives (Fake → Real): {false_negatives_mask.sum()} ({false_negatives_mask.sum() / len(y_val) * 100:.2f}%)")

print(f"\n🎯 Key Insights:")
if false_positives_mask.sum() > false_negatives_mask.sum():
    print(f"   → Model tends to over-predict FAKE news")
    print(f"   → More conservative (skeptical) predictions")
elif false_negatives_mask.sum() > false_positives_mask.sum():
    print(f"   → Model tends to under-predict FAKE news")
    print(f"   → Model is too trusting (optimistic)")
else:
    print(f"   → Model is balanced in both directions")

print(f"\n💡 Recommendations:")
if false_negatives_mask.sum() > false_positives_mask.sum():
    print(f"   1. Tune decision threshold to be more conservative")
    print(f"   2. Add more training examples of fake news")
    print(f"   3. Use ensemble methods to catch more fake news")
else:
    print(f"   1. Reduce false alarms by adjusting threshold")
    print(f"   2. Improve robustness of real news detection")
    print(f"   3. Check for potential bias in model")

print(f"\n✅ Error Analysis Complete!")